In [1]:
import pandas as pd
import numpy as np
import pickle
import re

In [2]:
with open(r"C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\Master Dataset.pkl", 'rb') as file:
    df = pickle.load(file)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45379 entries, 0 to 45378
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                45379 non-null  int64  
 1   genres                45379 non-null  object 
 2   id                    45379 non-null  int64  
 3   original_language     45379 non-null  object 
 4   original_title        45379 non-null  object 
 5   overview              45379 non-null  object 
 6   popularity            45379 non-null  float64
 7   production_companies  45378 non-null  object 
 8   production_countries  45378 non-null  object 
 9   release_date          45294 non-null  object 
 10  revenue               45378 non-null  float64
 11  runtime               45379 non-null  float64
 12  spoken_languages      45378 non-null  object 
 13  status                45297 non-null  object 
 14  title                 45379 non-null  object 
 15  vote_average       

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [4]:
df['combined_text'] = df['combined_text'].apply(clean_text)

In [5]:
from sentence_transformers import SentenceTransformer
import torch
import os

torch.set_num_threads(8)

model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    device='cpu'
)

text = df['combined_text'].tolist()

embeddings = model.encode(
    text,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/355 [00:00<?, ?it/s]

In [6]:
embeddings.shape

(45379, 384)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)
print(similarity.shape)
print(similarity[:5, :5])

(45379, 45379)
[[0.9999999  0.39163202 0.21678738 0.03550025 0.24980539]
 [0.39163202 1.0000001  0.28577602 0.09983586 0.25605237]
 [0.21678738 0.28577602 0.9999999  0.27389508 0.3262639 ]
 [0.03550025 0.09983586 0.27389508 1.0000002  0.31043965]
 [0.24980539 0.25605237 0.3262639  0.31043965 1.        ]]


In [8]:
movie_name = "Inception"

matches = df[df['title'].str.lower() == movie_name.lower()]

if len(matches) == 0:
    print("Movie not found")
else:
    idx = matches.index[0]
    print("Index:", idx)

Index: 15470


In [9]:
scores = similarity[idx]
print(scores[:10])

[0.20517075 0.20933554 0.24030644 0.30539978 0.21055773 0.38217393
 0.1412274  0.2610842  0.31612065 0.41165233]


In [10]:
similar_movies = list(enumerate(scores))

# sort by similarity descending
similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

# top 10 including itself
for i in similar_movies[:10]:
    print(df.iloc[i[0]]['title'], ":", round(i[1], 3))

Inception : 1.0
TRON: Legacy : 0.558
Demonlover : 0.541
Mission: Impossible : 0.539
Alone in the Dark : 0.537
Setup : 0.531
Mission: Impossible III : 0.526
Criminal : 0.526
The Insider : 0.525
Dreamscape : 0.523


In [11]:
print(df['combined_text'].head(10).tolist())

['toy story toy story animation comedy family animation comedy family jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life tom hanks tim allen don rickles jim varney wallace shawn john ratzenberger annie potts john morris erik von detten laurie metcalf r lee ermey sarah freeman penn jillette john lasseter in a world where toys are living things who pretend to be lifeless when humans are present a group of toys owned by six year old andy davis are caught off guard when andy s birthday party is moved up a week as andy his mother and infant sister molly are preparing to move the following week the toys leader and andy s favorite toy a pull string cowboy doll named sheriff woody organizes the other toys including bo peep the shepherdess mr potato head rex the dinosaur hamm the piggy bank and slinky dog into a scouting mission green army men led by sarge spy on the party and report the results to the others via baby monitors the toys are relieved when the part

In [12]:
df['combined_text'][0]

'toy story toy story animation comedy family animation comedy family jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life tom hanks tim allen don rickles jim varney wallace shawn john ratzenberger annie potts john morris erik von detten laurie metcalf r lee ermey sarah freeman penn jillette john lasseter in a world where toys are living things who pretend to be lifeless when humans are present a group of toys owned by six year old andy davis are caught off guard when andy s birthday party is moved up a week as andy his mother and infant sister molly are preparing to move the following week the toys leader and andy s favorite toy a pull string cowboy doll named sheriff woody organizes the other toys including bo peep the shepherdess mr potato head rex the dinosaur hamm the piggy bank and slinky dog into a scouting mission green army men led by sarge spy on the party and report the results to the others via baby monitors the toys are relieved when the party

In [13]:
pickle.dump(embeddings, open(r'C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\BERT\movie_embeddings.pkl', 'wb'))
pickle.dump(df, open(r'C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\BERT\processed_movies.pkl', 'wb'))